In [ ]:
import pandas as pd
from pymongo import MongoClient
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv()

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["my_project"]
collection = db["ddproperty_clean"]


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ddproperty"
df = pd.read_csv(PROCESSED_DIR/'ddproperty_processed.csv')

csv_ids = df["url"].astype(str).tolist()

existing = collection.find(
    {"url": {"$in": csv_ids}},
    {"url": 1, "_id": 0}
)

existing_ids = {item["url"] for item in existing}
df_unique = df[~df["url"].astype(str).isin(existing_ids)]

data_dict = df_unique.to_dict("records")

if len(data_dict) > 0:
    try:
        collection.insert_many(data_dict)
        print("Insert สำเร็จ")
    except Exception as e:
        print("เกิดข้อผิดพลาดขณะ insert:", e)
else:
    print("ไม่มีข้อมูลใหม่ให้เพิ่ม (ทั้งหมดซ้ำ)")

In [6]:
import pandas as pd
from pymongo import MongoClient
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
connection_string = MONGO_URI
client = MongoClient(connection_string)

db = client["my_project"]
collection = db["ddproperty_clean"]
cursor = collection.find({})

df = pd.DataFrame(list(cursor))

if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ddproperty"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR/"condo.csv"

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved CSV to:", OUTPUT_PATH)

Saved CSV to: D:\dsdengdeng-project-dsde\data\processed\ddproperty\condo.csv
